# Графы


**Граф** — множество **вершин** (nodes/vertices) и **рёбер** (edges) между ними.

Классификация:
- **Ориентированный (directed)** — рёбра имеют направление.
- **Неориентированный (undirected)** — нет направления.
- **Взвешенный (weighted)** — у каждого ребра есть вес/стоимость.
- **Циклический / ациклический (DAG)**.
- **Связный / несвязный**.


## Способы представления

Пусть `V` — число вершин, `E` — число рёбер.

| Способ | Память | Поиск ребра | Соседи |
|---|---|---|---|
| Список смежности (adjacency list) | O(V + E) | O(V) | O(deg(v)) |
| Матрица смежности (adjacency matrix) | O(V²) | O(1) | O(V) |
| Список рёбер (edge list) | O(E) | O(E) | O(E) |

**В 90% случаев на собесе** — список смежности через `dict`.


### Список смежности (adjacency list)


In [ ]:
# Неориентированный граф
graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D'],
    'C': ['A', 'D'],
    'D': ['B', 'C'],
}

# Ориентированный граф
directed = {
    'A': ['B', 'C'],
    'B': ['D'],
    'C': [],
    'D': [],
}

# Взвешенный ориентированный граф: (сосед, вес)
weighted = {
    'A': [('B', 5), ('C', 2)],
    'B': [('D', 4)],
    'C': [('D', 7)],
    'D': [],
}

### Матрица смежности (adjacency matrix)


In [ ]:
vertices = ['A', 'B', 'C']
idx = {v: i for i, v in enumerate(vertices)}
n = len(vertices)

matrix = [[0] * n for _ in range(n)]
edges = [('A', 'B'), ('A', 'C'), ('B', 'C')]
for u, v in edges:
    i, j = idx[u], idx[v]
    matrix[i][j] = 1
    matrix[j][i] = 1   # неориентированный

for row in matrix: print(row)

## BFS — поиск в ширину (Breadth-First Search)

Обходит граф **слоями** от стартовой вершины. Использует **очередь**.

Сложность: `O(V + E)`.

**Применения:**
- кратчайший путь в **невзвешенном** графе (по числу рёбер);
- проверка достижимости;
- обход уровней в дереве;
- поиск компонент связности.


In [ ]:
from collections import deque

def bfs(graph, start):
    visited = {start}
    queue = deque([start])
    order = []
    while queue:
        v = queue.popleft()
        order.append(v)
        for neighbor in graph[v]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return order

graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [], 'E': [], 'F': [],
}
bfs(graph, 'A')   # ['A', 'B', 'C', 'D', 'E', 'F']

### BFS для поиска кратчайшего пути


In [ ]:
from collections import deque

def shortest_path(graph, start, end):
    if start == end:
        return [start]
    visited = {start}
    queue = deque([(start, [start])])
    while queue:
        v, path = queue.popleft()
        for neighbor in graph[v]:
            if neighbor in visited:
                continue
            new_path = path + [neighbor]
            if neighbor == end:
                return new_path
            visited.add(neighbor)
            queue.append((neighbor, new_path))
    return None

shortest_path(graph, 'A', 'F')   # ['A', 'C', 'F']

## DFS — поиск в глубину (Depth-First Search)

Идём как можно глубже, потом откатываемся. Использует **стек** (рекурсию).

Сложность: `O(V + E)`.

**Применения:**
- топологическая сортировка;
- проверка ацикличности;
- поиск сильно связных компонент;
- backtracking-задачи (лабиринт, sudoku).


In [ ]:
def dfs_recursive(graph, start, visited=None, order=None):
    if visited is None:
        visited, order = set(), []
    visited.add(start)
    order.append(start)
    for neighbor in graph[start]:
        if neighbor not in visited:
            dfs_recursive(graph, neighbor, visited, order)
    return order

dfs_recursive(graph, 'A')   # ['A', 'B', 'D', 'E', 'C', 'F']

### DFS итеративный (на явном стеке)


In [ ]:
def dfs_iterative(graph, start):
    visited = set()
    stack = [start]
    order = []
    while stack:
        v = stack.pop()
        if v in visited:
            continue
        visited.add(v)
        order.append(v)
        # добавляем соседей в обратном порядке для естественного обхода
        for neighbor in reversed(graph[v]):
            if neighbor not in visited:
                stack.append(neighbor)
    return order

dfs_iterative(graph, 'A')

## BFS vs DFS

| | BFS | DFS |
|---|---|---|
| Структура | очередь | стек / рекурсия |
| Поведение | по слоям | вглубь, потом назад |
| Кратчайший путь (невзвеш.) | ✅ | ❌ |
| Топологическая сортировка | ❌ | ✅ |
| Память | O(ширины) | O(глубины) |


## Алгоритм Дейкстры — кратчайший путь во взвешенном графе

Работает на графе с **неотрицательными** весами.

Идея: на каждом шаге из непосещённых вершин берём ту, до которой текущая известная дистанция минимальна (через **min-heap**), и **релаксируем** её рёбра.

Сложность: `O((V + E) · log V)` с кучей.


In [ ]:
import heapq

def dijkstra(graph, start):
    """
    graph: {node: [(neighbor, weight), ...]}
    Возвращает: {node: min_distance}
    """
    distances = {node: float('inf') for node in graph}
    distances[start] = 0
    heap = [(0, start)]   # (distance, node)

    while heap:
        d, u = heapq.heappop(heap)
        if d > distances[u]:
            continue                # устаревшая запись
        for v, w in graph[u]:
            new_d = d + w
            if new_d < distances[v]:
                distances[v] = new_d
                heapq.heappush(heap, (new_d, v))
    return distances

weighted_graph = {
    'A': [('B', 5), ('C', 2)],
    'B': [('D', 4)],
    'C': [('B', 1), ('D', 7)],
    'D': [],
}
dijkstra(weighted_graph, 'A')
# {'A': 0, 'B': 3, 'C': 2, 'D': 7}

### Почему Дейкстра не работает с отрицательными весами?
Алгоритм опирается на то, что **уже посещённую вершину не нужно пересматривать**. Если в графе появится отрицательное ребро, путь через уже «закрытую» вершину может оказаться короче — но мы это пропустим.

Для отрицательных весов — **Bellman-Ford** (`O(V·E)`).


## Топологическая сортировка
Только для **DAG** (направленный ацикличный граф). Упорядочивает вершины так, что для любого ребра `u → v` вершина `u` идёт раньше `v`.

Применения: порядок сборки задач, разрешение зависимостей, расписания.


In [ ]:
from collections import deque

def topological_sort(graph):
    # граф ориентированный: graph[u] = [v1, v2, ...]
    in_degree = {v: 0 for v in graph}
    for u in graph:
        for v in graph[u]:
            in_degree[v] = in_degree.get(v, 0) + 1
            in_degree.setdefault(u, in_degree[u])

    queue = deque([v for v, d in in_degree.items() if d == 0])
    order = []
    while queue:
        u = queue.popleft()
        order.append(u)
        for v in graph.get(u, []):
            in_degree[v] -= 1
            if in_degree[v] == 0:
                queue.append(v)

    if len(order) != len(in_degree):
        raise ValueError('Цикл в графе')
    return order

tasks = {
    'wake_up': ['breakfast'],
    'breakfast': ['work'],
    'shower': ['breakfast', 'work'],
    'work': [],
}
topological_sort(tasks)

## Что спрашивают на собеседовании

- В чём разница между **BFS и DFS** и где что применять?
- Как **обнаружить цикл** в графе? *(В неорграфе — DFS + проверка повторного посещения через родителя. В орграфе — DFS с тремя состояниями: white/gray/black.)*
- Когда **Dijkstra**, а когда **Bellman-Ford**?
- Как решить задачу про **зависимости задач** (порядок выполнения)? *(Топосорт.)*
- В чём разница между **adjacency list** и **adjacency matrix**? Когда что выбирать?
